# IncuBrix Track 03: Open-Source Draft Video Generation and Model Routing
### Interactive Execution & Evaluation Notebook (Google Colab / Kaggle / Local CPU Host)

This notebook demonstrates the end-to-end open-source draft video generation system:
1. **Capability Registry & Routing**: Evaluates 5 open-weight models with fallback chains.
2. **Deterministic Scene Planning**: Generates pacing, beats, and visual prompts for Education, News, and Product archetypes.
3. **Audio & Subtitle Synthesis**: Generates narration and synchronized SRT / WebVTT timed captions.
4. **Reliable Clip Generation**: Demonstrates synthesis with controlled retries and CPU procedural fallback.
5. **FFmpeg Timeline Assembly**: Produces broadcast-standard 16:9 and 9:16 MP4 video drafts.
6. **Objective Quality Gate**: Validates stream integrity, decodability, duration precision, and cross-artifact contracts.
7. **Cryptographic Provenance**: Generates SHA-256 asset manifests conforming to assessment requirements.

## 1. Environment Inspection & Setup
Verify runtime hardware, Python version, and FFmpeg installation.

In [ ]:
import sys
import os
import platform
import shutil

print(f"Python Version: {sys.version}")
print(f"Platform:       {platform.system()} {platform.machine()}")

# Check FFmpeg
ffmpeg_bin = shutil.which("ffmpeg")
print(f"FFmpeg Binary:  {ffmpeg_bin or 'Missing - run apt-get install ffmpeg'}")
if not ffmpeg_bin and platform.system() == 'Linux':
    !apt-get update -qq && apt-get install -y -qq ffmpeg

## 2. Package Installation
Install dependencies and import the core `video_draft` package.

In [ ]:
!pip install -q click pydantic pyyaml soundfile numpy Pillow pytest
import video_draft
print(f"video_draft loaded from: {video_draft.__file__}")

## 3. Creative Brief Loading & Inspection
Load a standardized Creative Brief defining genre, aspect ratio, duration, and script.

In [ ]:
from pathlib import Path
import json
from video_draft.schema.brief import CreativeBrief

brief_file = Path("configs/briefs/baseline_16x9.json")
if brief_file.is_file():
    brief = CreativeBrief(**json.loads(brief_file.read_text(encoding='utf-8')))
else:
    # Fallback to in-memory definition
    from video_draft.schema.brief import SystemConstraints
    brief = CreativeBrief(
        id="brief-colab-demo",
        title="Cellular Mitosis Explainer",
        genre="education",
        aspect_ratio="16:9",
        target_duration_sec=16.0,
        script="Mitosis is cell division where chromosomes replicate and divide.",
        seed=42,
        constraints=SystemConstraints(max_vram_gb=16.0, allow_cpu_fallback=True, quality_tier='draft'),
    )

print(f"Loaded Brief: {brief.id} ({brief.title})")
print(f"Genre: {brief.genre} | Aspect Ratio: {brief.aspect_ratio} | Duration: {brief.target_duration_sec}s")

## 4. Capability-Based Routing Engine
Evaluate open-source video models and compute explainable ranking with deterministic tie-breaking.

In [ ]:
from video_draft.router.router import route

decision = route(creative_brief=brief, force_cpu=True)
print(f"Selected Model:   {decision.selected_model_name} (ID: {decision.selected_model})")
print(f"Target Hardware:  {decision.target_hardware.upper()}")
print(f"Selection Score:  {decision.selected_score:.4f}")
print(f"Fallback Chain:   {' -> '.join(decision.fallback_chain)}")
print(f"Rationale:        {decision.rationale}")

## 5. Deterministic Scene Planning
Decompose creative brief into structured scene beats with pacing tempo and visual prompt guidance.

In [ ]:
from video_draft.planner.scene_planner import ScenePlanner

planner = ScenePlanner()
plan = planner.plan(brief)
print(f"Plan ID:          {plan.plan_id}")
print(f"Strategy:         {plan.strategy_name} ({plan.genre})")
print(f"Planned Duration: {plan.planned_duration_sec:.2f}s across {len(plan.scenes)} beats
")
for s in plan.scenes:
    print(f"  [Scene {s.scene_index}] {s.start_sec:.1f}s - {s.end_sec:.1f}s: {s.title}")
    print(f"     Visual Prompt: {s.visual_prompt[:65]}...")

## 6. End-to-End Pipeline Run
Execute synthesis, audio muxing, subtitle creation, and timeline assembly via CLI.

In [ ]:
!video-draft run --brief configs/briefs/baseline_16x9.json --output-dir outputs/colab_run --mock

## 7. Inspect Quality Gate Scorecard
Evaluate stream probe, bitstream decodability, duration precision, and cross-artifact consistency.

In [ ]:
!video-draft evaluate --manifest outputs/colab_run/manifest.json

## 8. Inspect Generated Captions (.srt)
View the timed SubRip caption cues generated directly from the scene plan.

In [ ]:
srt_file = Path("outputs/colab_run/captions.srt")
if srt_file.is_file():
    print(srt_file.read_text(encoding='utf-8'))
else:
    print("Captions not found. Run cell 6 first.")

## 9. In-Notebook Video Playback
Render the assembled draft MP4 video directly within the notebook.

In [ ]:
from IPython.display import Video, display
vid_path = "outputs/colab_run/final_draft.mp4"
if os.path.exists(vid_path):
    display(Video(vid_path, embed=True, width=640))
else:
    print("Video file not found at:", vid_path)

## 10. Run Automated Pipeline Benchmark
Measure stage-by-stage latencies and test synthetic reliability across archetypes.

In [ ]:
!video-draft benchmark --runs 3 --synthetic